In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed
from skrebate import ReliefF  # Import ReliefF from scikit-rebate

# Load the dataset
# Note: Ensure that "class1_dataset.xlsx" is in your working directory or provide the correct path.
data = pd.read_excel("class1_dataset.xlsx")

# Extract the predictors and outcome
X = data.drop('RRI', axis=1)
Y = data['RRI']

# Define a list of numbers of top features to evaluate
n_features_list = list(range(1, X.shape[1] + 1))  # From 1 to total number of features

# Define a single Logistic Regression classifier with default hyperparameters
classifiers = [
    ("Logistic Default", LogisticRegression(random_state=42, max_iter=1000))
]

# Dictionary to store the best AUC, feature combination, and number of features for each classifier
best_results_by_classifier = {name: (0, None, None) for name, _ in classifiers}

def compute_best_result(n_features, classifier_name, model, X, Y, feature_scores, feature_indices_sorted):
    """
    Compute the best ROC AUC score for a given number of top features and classifier.

    Parameters:
    - n_features: Number of top features to select.
    - classifier_name: Name of the classifier.
    - model: The classifier instance.
    - X: All features.
    - Y: All labels.
    - feature_scores: Feature scores from RELIEF.
    - feature_indices_sorted: Indices of features sorted by their RELIEF scores.

    Returns:
    - Tuple containing classifier name, AUC, best feature indices, and number of features.
    """
    # Select the top 'n_features' based on RELIEF scores
    top_features_indices = feature_indices_sorted[:n_features]
    X_reduced = X.iloc[:, top_features_indices]

    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    aucs = []
    for train_index, test_index in skf.split(X_reduced, Y):
        X_train_fold, X_test_fold = X_reduced.iloc[train_index], X_reduced.iloc[test_index]
        y_train_fold, y_test_fold = Y.iloc[train_index], Y.iloc[test_index]
        model.fit(X_train_fold, y_train_fold)
        if hasattr(model, "predict_proba"):
            y_pred_probs_fold = model.predict_proba(X_test_fold)[:, 1]
        else:
            # For classifiers that do not have predict_proba, use decision_function
            y_pred_probs_fold = model.decision_function(X_test_fold)
            # Scale the decision function to [0,1] using min-max scaling
            y_pred_probs_fold = (y_pred_probs_fold - y_pred_probs_fold.min()) / (y_pred_probs_fold.max() - y_pred_probs_fold.min() + 1e-8)
        aucs.append(roc_auc_score(y_test_fold, y_pred_probs_fold))

    mean_auc = np.mean(aucs)
    return (classifier_name, mean_auc, top_features_indices, n_features)

# Initialize RELIEF for feature ranking
relief = ReliefF(n_neighbors=100, n_features_to_select=X.shape[1], discrete_threshold=10, verbose=True, n_jobs=7)
relief.fit(X.values, Y.values)

# Get feature scores and sort feature indices by score in descending order
feature_scores = relief.feature_importances_
feature_indices_sorted = np.argsort(feature_scores)[::-1]  # Indices of features sorted by importance

# Create a list of all (n_features, classifier) pairs with data
tasks = [(n_features, name, model, X, Y, feature_scores, feature_indices_sorted) 
         for n_features in n_features_list for name, model in classifiers]

# Parallelize the computation across available cores
results = Parallel(n_jobs=7)(
    delayed(compute_best_result)(n_features, name, model, X, Y, feature_scores, feature_indices_sorted) 
    for n_features, name, model, X, Y, feature_scores, feature_indices_sorted in tasks
)

# Aggregate results to find the best for each classifier
for classifier_name, auc, features, n_features in results:
    current_best_auc, _, _ = best_results_by_classifier[classifier_name]
    if auc > current_best_auc:
        best_results_by_classifier[classifier_name] = (auc, features, n_features)

# The final results are stored in the 'best_results_by_classifier' dictionary.
print("Best Results by Classifier:")
for clf_name, (auc, features, n_feats) in best_results_by_classifier.items():
    print(f"Classifier: {clf_name}")
    print(f"  Best AUC: {auc:.4f}")
    print(f"  Number of Features: {n_feats}")
    print(f"  Feature Indices: {features}\n")

Created distance array in 1.406867504119873 seconds.
Feature scoring under way ...
Completed scoring in 37.86457681655884 seconds.
Best Results by Classifier:
Classifier: Logistic Default
  Best AUC: 0.6604
  Number of Features: 39
  Feature Indices: [37 30 25 26 29 28 32 38 36 10  7 13 33 12 34 24 14 18  3 15  6 16  8 11
 17  4  1  9 19 31  5  0 20 35 22 23  2 27 21]

